In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
tf.keras.backend.clear_session()

2026-04-13 05:39:36.322170: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776058776.529988      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776058776.587099      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776058777.072062      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776058777.072102      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776058777.072105      55 computation_placer.cc:177] computation placer alr

In [3]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121, ResNet50

input_orig = layers.Input(shape=(224,224,3), name="/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset")
input_vessel = layers.Input(shape=(224,224,3), name="/kaggle/input/datasets/amimulahasanrofik/vesselenhancedmaps")
input_tab = layers.Input(shape=(Xt_train.shape[1],), name="/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv")

ValueError: Argument `name` must be a string and cannot contain character `/`. Received: name=/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset (of type <class 'str'>)

In [ ]:
# Original branch
base1 = DenseNet121(weights='imagenet', include_top=False)
x1 = base1(input_orig)   # ✅ attach manually

# Vessel branch
base2 = ResNet50(weights='imagenet', include_top=False)
x2 = base2(input_vessel)  # ✅ attach manually

In [ ]:
for layer in base1.layers:
    layer.trainable = False

for layer in base2.layers:
    layer.trainable = False

In [ ]:
x1 = layers.GlobalAveragePooling2D()(x1)
x1 = layers.BatchNormalization()(x1)

x2 = layers.GlobalAveragePooling2D()(x2)
x2 = layers.BatchNormalization()(x2)

In [ ]:
x3 = layers.Dense(64, activation='relu')(input_tab)
x3 = layers.BatchNormalization()(x3)
x3 = layers.Dropout(0.3)(x3)

x3 = layers.Dense(32, activation='relu')(x3)

In [ ]:
combined = layers.Concatenate()([x1, x2, x3])

# Safe attention
attn = layers.Dense(256, activation='relu')(combined)
attn = layers.Dense(combined.shape[-1], activation='sigmoid')(attn)

combined = layers.Multiply()([combined, attn])

In [ ]:
x = layers.Dense(512, activation='relu')(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.6)(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation='relu')(x)

output = layers.Dense(num_classes, activation='softmax')(x)

In [ ]:
model = models.Model(
    inputs=[input_orig, input_vessel, input_tab],
    outputs=output
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    [Xo_train, Xv_train, Xt_train],
    y_train,
    validation_data=([Xo_test, Xv_test, Xt_test], y_test),
    epochs=20,
    batch_size=8
)

In [ ]:
for layer in base1.layers[-20:]:
    layer.trainable = True

for layer in base2.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    [Xo_train, Xv_train, Xt_train],
    y_train,
    validation_data=([Xo_test, Xv_test, Xt_test], y_test),
    epochs=10,
    batch_size=8
)